# 🌬️ Конвеєр обробки даних якості повітря
## Від сирих сенсорних даних до ознак для машинного навчання

Цей ноутбук реалізує повний конвеєр **ETL + машинне навчання** для даних моніторингу якості повітря,
отриманих з Українського порталу відкритих даних. Архітектура побудована навколо принципу **впровадження залежностей**:
кожна функція отримує свої залежності (DataFrame, сесії, конфігураційні словники) явно як аргументи —
що робить кожен етап незалежно тестованим і придатним для повторного використання без прихованого глобального стану.

### Огляд конвеєра
```
Веб-скрапінг → Очищення даних → Схема БД → ETL-завантаження → EDA → Побудова ознак → Навчання моделей
```

---
> **Архітектурна примітка:** Конвеєр використовує схему сховища даних *Snowflake* внутрішньо,
> а потім перетворює дані у широкий, ресемпльований формат для машинного навчання.


## 1. 📦 Імпорти та залежності

Всі бібліотеки імпортуються на початку. Групування за призначенням дозволяє виявити відсутні залежності до запуску будь-яких наступних клітин.


In [ ]:
import os
import re
from pathlib import Path
from datetime import datetime

# ── Numerical & tabular ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# ── Database (SQLAlchemy ORM) ─────────────────────────────────────────────────
from sqlalchemy import (
    Column, Integer, BigInteger, Float,
    String, Boolean, DateTime, ForeignKey
)
from sqlalchemy import create_engine, URL
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker

# ── Machine Learning ──────────────────────────────────────────────────────────
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# ── Utilities ─────────────────────────────────────────────────────────────────
from typing import Dict, Any, List, Tuple
from IPython.display import clear_output, display

# ── Web scraping ──────────────────────────────────────────────────────────────
from bs4 import BeautifulSoup
import requests

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid")


## 2. ⚙️ Центральна конфігурація

Замість того щоб розкидати магічні числа та шляхи по всьому ноутбуку, **всі параметри зосереджені в одному словнику**.
Функції отримують цей конфіг через впровадження залежностей, тому зміна значення тут автоматично поширюється скрізь.


In [ ]:
CONFIG: Dict[str, Any] = {
    # ── Data sources ──────────────────────────────────────────────────────────
    "source_url"    : "https://opendata.gov.ua/dataset/air_monitor",
    "data_dir"      : "./data",
    "date_regex"    : r"\d{4}-\d{2}-\d{2}",

    # ── Database connection ────────────────────────────────────────────────────
    "db_driver"     : "mysql",
    "db_host"       : "db",       # replace with 127.0.0.1 for local MySQL
    "db_port"       : 3306,

    # ── ETL ────────────────────────────────────────────────────────────────────
    "batch_size"    : 5000,

    # ── ML pipeline ────────────────────────────────────────────────────────────
    "resample_freq" : "h",         # hourly resampling
    "forecast_horizon_h": 3,       # predict N hours ahead
    "pm25_clip_upper"   : 300,     # µg/m³ — WHO extreme threshold
    "pm10_clip_upper"   : 500,
    "interp_limit"      : 3,       # max hours to linearly interpolate
    "min_station_hours" : 1500,    # stations with fewer hours are dropped
    "min_continuity"    : 0.5,     # fraction of non-NaN hours required
    "top_k_neighbors"   : 3,       # spatial neighbours for gap-filling
    "test_size"         : 0.2,
    "cv_splits"         : 5,
    "use_pm10_feature"  : False,   # toggle PM10 as a predictor
}


## 3. 🕸️ Збір даних (веб-скрапінг)

Вихідні CSV-файли опубліковані на Українському порталі відкритих даних. Цей розділ аналізує HTML-сторінку порталу,
витягує дату та URL завантаження кожного файлу, а потім записує файли до `./data/`.

> **Впровадження залежностей:** `scrape_air_monitor_data` отримує URL та цільову директорію як аргументи,
> що дозволяє легко перенаправити на дзеркало або тестовий сервер без редагування тіла функції.


In [ ]:
def scrape_air_monitor_data(source_url: str, data_dir: str, date_regex: str) -> None:
    """
    Завантажує всі CSV-файли, перелічені на сторінці відкритих даних моніторингу повітря.

    Parameters
    ----------
    source_url : str -- URL сторінки індексу набору даних.
    data_dir   : str -- локальна директорія для збереження файлів.
    date_regex : str -- регулярний вираз для витягування рядка дати з назви файлу.
    """
    Path(data_dir).mkdir(parents=True, exist_ok=True)

    response = requests.get(source_url)
    soup = BeautifulSoup(response.text, "html.parser")

    for item in soup.find_all("div", {"class": "resource-item"}):
        # ── Пошук назви файлу для відображення ───────────────────────────────
        name_block = item.find("div", {"class": "data-resource-name-content"})
        link_tag   = name_block.find("a") if name_block else None
        if not link_tag:
            continue

        # ── Витягування підрядка дати для імені файлу ────────────────────────
        raw_title = link_tag.string or ""
        matches   = re.findall(date_regex, raw_title)
        if not matches:
            continue
        filename = Path(data_dir) / f"{matches[0]}.csv"

        # ── Пошук посилання для завантаження ─────────────────────────────────
        download_tag = item.find("a", {"class": "data-resource-download"}, href=True)
        if not download_tag:
            continue
        table_link = download_tag.get("href")

        # ── Завантаження і збереження ────────────────────────────────────────
        file_response = requests.get(table_link)
        print(f"[{file_response.status_code}] {filename.name}  ← {table_link}")

        if file_response.status_code == 200:
            filename.write_bytes(file_response.content)


# Запуск скрапера (закоментувати, якщо файли вже завантажено)
# scrape_air_monitor_data(
#     source_url=CONFIG["source_url"],
#     data_dir=CONFIG["data_dir"],
#     date_regex=CONFIG["date_regex"],
# )


## 4. 📂 Завантаження та об'єднання даних

### 4.1 Перевірка сумісності стовпців

Перед об'єднанням усіх CSV-файлів перевіряється, що кожен файл має **однакову схему стовпців**.
Якщо новий експорт додасть або перейменує стовпець, перевірка виявить це негайно, запобігаючи мовчазному некоректному об'єднанню.


In [ ]:
def check_if_columns_same(data_dir: str) -> bool:
    """
    Повертає True, якщо кожен CSV у data_dir має ідентичний набір назв стовпців.

    Parameters
    ----------
    data_dir : str -- директорія з вихідними CSV-файлами.
    """
    seen_schemas = set()

    for filepath in Path(data_dir).rglob("*.csv"):
        try:
            sample = pd.read_csv(filepath, nrows=0)          # header only
        except UnicodeDecodeError:
            sample = pd.read_csv(filepath, nrows=0, encoding="cp1251")

        seen_schemas.add(tuple(sample.columns))

    return len(seen_schemas) == 1


columns_match = check_if_columns_same(CONFIG["data_dir"])
print("Всі файли мають однакові стовпці:", columns_match)


### 4.2 Об'єднання всіх CSV-файлів

Оскільки всі файли мають однакову схему, їх можна безпечно конкатенувати за допомогою `pd.concat`.


In [ ]:
def combine_tables(data_dir: str) -> pd.DataFrame:
    """
    Зчитує всі CSV-файли з data_dir і конкатенує їх в єдиний DataFrame.

    Parameters
    ----------
    data_dir : str -- директорія з вихідними CSV-файлами.

    Returns
    -------
    pd.DataFrame -- конкатенований набір даних зі скинутим індексом.
    """
    files = list(Path(data_dir).rglob("*.csv"))
    frames = []
    for f in files:
        try:
            frames.append(pd.read_csv(f))
        except UnicodeDecodeError:
            frames.append(pd.read_csv(f, encoding="cp1251"))

    return pd.concat(frames, ignore_index=True)


df = combine_tables(CONFIG["data_dir"])
print(f"Розміри об'єднаного DataFrame: {df.shape}")
df.head()


## 5. 🧹 Очищення даних

Вихідні дані містять ряд проблем якості, які необхідно вирішити перед використанням:

| Проблема | Виправлення |
|---|---|
| Відсутні значення вимірювань / ідентифікаторів | Видалення відповідних рядків |
| Станції з кількома суперечливими назвами | Перейменування до канонічних назв |
| Станції з суперечливими GPS-координатами | Заміна на моду координат по станції |
| Нестандартні коди параметрів (`SDS_P1`, `PM25`, …) | Приведення до канонічних назв (`PM10`, `PM2.5`, …) |
| Нестандартні рядки одиниць вимірювання | Нормалізація до Unicode-символів (`µg/m³`, `°C`, …) |
| Змішані формати дати-часу (ISO 8601 та застарілі) | Парсинг з `format='mixed'` |


### 5.1 Видалення рядків з відсутніми критичними полями


In [ ]:
def prune_invalid_measurements(df: pd.DataFrame) -> pd.DataFrame:
    """
    Видаляє рядки, де відсутнє значення вимірювання або ідентифікатор параметра.

    Parameters
    ----------
    df : pd.DataFrame -- вихідний об'єднаний набір даних.

    Returns
    -------
    pd.DataFrame -- очищена копія з видаленими некоректними рядками.
    """
    df = df.copy()
    initial = len(df)

    df = df.dropna(subset=["stations_params_value", "stations_params_id"])

    dropped = initial - len(df)
    print(f"Видалено рядків  : {dropped:,}  ({dropped/initial:.1%})")
    print(f"Залишилось рядків: {len(df):,}")

    # Замінюємо NaN на None для коректної обробки SQLAlchemy
    return df.replace({float("nan"): None})


df = prune_invalid_measurements(df)


### 5.2 Вирішення конфліктів назв станцій

Один `stations_id` повинен мати рівно одну `stations_name`. Функція нижче виявляє будь-які порушення;
якщо вони знайдені, виконується перейменування до стабільної канонічної назви.


In [ ]:
def identify_duplicate_names(df: pd.DataFrame) -> None:
    """
    Виводить ідентифікатори станцій, пов'язаних з більш ніж однією назвою.

    Parameters
    ----------
    df : pd.DataFrame -- набір даних зі стовпцями 'stations_id' та 'stations_name'.
    """
    name_groups = df.groupby("stations_id")["stations_name"].unique()
    problematic = name_groups[name_groups.apply(len) > 1]

    if problematic.empty:
        print("✅ Перевірка назв пройшла: конфліктів не виявлено.")
        return

    print(f"⚠️  {len(problematic)} ідентифікатор(ів) станцій з конфліктуючими назвами:\n")
    for sid, names in problematic.items():
        print(f"  ID {sid}: {', '.join(map(str, names))}")
        print("  " + "-" * 40)


identify_duplicate_names(df)


In [ ]:
def remap_conflicting_names(df: pd.DataFrame, stations_mapping: Dict[int, str]) -> pd.DataFrame:
    """
    Замінює назви станцій для ідентифікаторів, присутніх у stations_mapping.

    Parameters
    ----------
    df               : pd.DataFrame -- набір даних для оновлення.
    stations_mapping : Dict[int, str] -- відображення {station_id: canonical_name}.

    Returns
    -------
    pd.DataFrame -- копія з виправленими назвами станцій.
    """
    df = df.copy()
    mask = df["stations_id"].isin(stations_mapping.keys())
    df.loc[mask, "stations_name"] = df.loc[mask, "stations_id"].map(stations_mapping)
    return df


# Відображення канонічних назв для станцій з конфліктуючими мітками
STATION_NAME_MAP: Dict[int, str] = {
    256: "vinnytsia-256",
    281: "vinnytsia-281",
    315: "vinnytsia-315",
     90: "vinnytsia-90",
    271: "vinnytsia-271",
    767: "Соборна 36",
   1183: "Вишенька",
    246: "vinnytsia-246",
    274: "vinnytsia-274",
}

df = remap_conflicting_names(df, STATION_NAME_MAP)
identify_duplicate_names(df)   # перевірка — має вивести ✅


### 5.3 Стандартизація GPS-координат

Деякі станції мають незначно різні координати в різних експортах (похибки округлення, помилки введення).
Замінюємо всі значення координат для кожної станції на **моду** (найбільш часто спостережуване значення) -- статистично стійкий показник для такого типу шуму.


In [ ]:
def check_coordinate_consistency(df: pd.DataFrame) -> None:
    """
    Виводить станції, чиї стовпці Lat або Long містять більше одного унікального значення.

    Parameters
    ----------
    df : pd.DataFrame -- набір даних зі стовпцями 'stations_id', 'Lat', 'Long'.
    """
    stats = df.groupby("stations_id")[["Lat", "Long"]].nunique()
    bad   = stats[(stats["Lat"] > 1) | (stats["Long"] > 1)].index

    if bad.empty:
        print("✅ Перевірка координат пройшла: всі станції мають узгоджені координати.")
        return

    print(f"⚠️  {len(bad)} станція(ї) з суперечливими координатами:\n")
    for sid in bad:
        lats  = df.loc[df["stations_id"] == sid, "Lat"].unique()
        longs = df.loc[df["stations_id"] == sid, "Long"].unique()
        print(f"  Station {sid} | Lat: {lats}  Long: {longs}")


def standardize_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищає, конвертує та нормалізує координати Lat/Long через моду.

    Parameters
    ----------
    df : pd.DataFrame -- набір даних для оновлення.

    Returns
    -------
    pd.DataFrame -- той самий DataFrame з очищеними стовпцями координат.
    """
    df = df.copy()

    def _get_mode(series: pd.Series):
        m = series.mode()
        return m.iloc[0] if not m.empty else None

    for col in ["Lat", "Long"]:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str).str.extract(r"(\d+\.\d+)")[0]
        df[col] = pd.to_numeric(df[col], errors="coerce").round(5)

    df[["Lat", "Long"]] = df.groupby("stations_id")[["Lat", "Long"]].transform(_get_mode)
    print("✅ Координати очищено та стандартизовано.")
    return df


check_coordinate_consistency(df)
df = standardize_coordinates(df)
check_coordinate_consistency(df)   # перевірка -- має вивести ✅


### 5.4 Стандартизація кодів параметрів та одиниць вимірювання

Сенсори різних виробників звітують про той самий забруднювач під різними кодами
(`SDS_P2`, `PM25`, `PM2.5`). Виконується приведення всього до єдиного канонічного словника.
Аналогічно, одиниці надходять у різних кодуваннях (`ug/m3`, `мкг/м³`) і нормалізуються
до правильних Unicode-символів (`µg/m³`).


In [ ]:
# ── Канонічний код параметра → назва для відображення ───────────────────────
CANONICAL_PARAMS: Dict[str, str] = {
    # Тверді частинки
    "SDS_P1": "PM10",   "SDS_P2": "PM2.5",
    "PMS_P0": "PM1.0",  "PMS_P1": "PM10",   "PMS_P2": "PM2.5",
    "PM0"   : "PM1.0",  "PM1"   : "PM1.0",  "PM25"  : "PM2.5",
    "PM100" : "PM10",   "PM1.0" : "PM1.0",  "PM2.5" : "PM2.5",
    "PM10"  : "PM10",
    # Гази
    "CO2": "CO2", "CO": "CO", "NO2": "NO2", "NO₂": "NO2",
    "O3" : "O3",  "O₃": "O3", "NH3": "NH3",
    "CH2O": "HCHO", "H2CO": "HCHO", "VOC": "VOC",
    # Метеорологічні параметри
    "TEMPERATURE": "Temperature",
    "HUMIDITY"   : "Humidity",
    "PRESSURE"   : "Pressure",
    # Радіація
    "RAD": "Radiation",
    # Псевдоніми для конкретних сенсорів
    "A4": "CO", "E1": "NO2", "E3": "O3",
}

# ── Нестандартний рядок одиниць → Unicode-канонічна форма ───────────────────
UNIT_MAP: Dict[str, str] = {
    "ug/m3"   : "µg/m³", "мкг/м³": "µg/m³", "ug/m³": "µg/m³",
    "ppm"     : "ppm",   "ppb"    : "ppb",
    "%"       : "%",     "Rh"     : "%",
    "°C"      : "°C",   "C"      : "°C",
    "Pa"      : "Pa",   "hPa"    : "hPa",
    "mg/m3"   : "mg/m³",
    "uSv/h"   : "µSv/h",
}


In [ ]:
def resolve_canonical_parameter(raw_key: Any) -> str:
    """
    Перетворює сирий код параметра сенсора на канонічну назву.

    Parameters
    ----------
    raw_key : Any -- сире значення зі стовпця 'stations_params_key'.

    Returns
    -------
    str -- канонічна назва або рядок у верхньому регістрі, якщо відображення відсутнє.
    """
    if pd.isna(raw_key) or str(raw_key).lower() == "nan":
        return "UNKNOWN"

    key = str(raw_key).strip()

    # Видалення всього до знаку '=' (префікс виробника)
    if "=" in key:
        key = key.split("=")[-1]

    # Видалення вмісту в дужках (одиниці, вбудовані в код)
    key = re.sub(r"\(.*?\)", "", key)

    # Нормалізація роздільників та регістру
    key = key.replace(" ", "").replace("_", "").replace(".", "").upper()

    return CANONICAL_PARAMS.get(key, key)


def unify_measurement_units(df: pd.DataFrame, unit_map: Dict[str, str]) -> pd.DataFrame:
    """
    Нормалізує стовпець 'stations_params_unit' до Unicode-канонічних символів.

    Parameters
    ----------
    df       : pd.DataFrame -- набір даних для обробки.
    unit_map : Dict[str, str] -- відображення сирих рядків одиниць до канонічних.

    Returns
    -------
    pd.DataFrame -- копія з нормалізованим стовпцем одиниць.
    """
    def _clean(u):
        if pd.isna(u):
            return "unknown"
        return unit_map.get(str(u).strip().replace("(", "").replace(")", ""), str(u).strip())

    df = df.copy()
    df["stations_params_unit"] = df["stations_params_unit"].apply(_clean)
    return df


def standardize_dimension_attributes(
    df        : pd.DataFrame,
    canonical : Dict[str, str],
    unit_map  : Dict[str, str],
) -> pd.DataFrame:
    """
    Оркеструє нормалізацію кодів параметрів та одиниць вимірювання.

    Parameters
    ----------
    df        : pd.DataFrame -- набір даних для очищення.
    canonical : Dict[str, str] -- словник CANONICAL_PARAMS.
    unit_map  : Dict[str, str] -- словник UNIT_MAP.

    Returns
    -------
    pd.DataFrame -- повністю нормалізована копія.
    """
    df = df.copy()
    df["stations_params_key"] = df["stations_params_key"].apply(resolve_canonical_parameter)
    df = unify_measurement_units(df, unit_map)
    # Заповнення відсутніх назв канонічним кодом
    df["stations_params_name"] = df["stations_params_name"].fillna(df["stations_params_key"])
    return df


df = standardize_dimension_attributes(df, CANONICAL_PARAMS, UNIT_MAP)
print("Унікальні коди параметрів після нормалізації:")
print(sorted(df["stations_params_key"].unique()))


### 5.5 Парсинг та уніфікація міток часу

Два стовпці міток часу (`stations_time` та `stations_params_time`) містять суміш ISO 8601 та застарілих форматів.
`format='mixed'` Pandas обробляє обидва прозоро та конвертує все в UTC.


In [ ]:
def prepare_dataframe_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Парсить обидва стовпці дати-часу до UTC-усвідомлених міток pd.Timestamp.

    Parameters
    ----------
    df : pd.DataFrame -- вихідний або частково очищений DataFrame.

    Returns
    -------
    pd.DataFrame -- копія з обома стовпцями дати, конвертованими до UTC.
    """
    df = df.copy()
    for col in ["stations_time", "stations_params_time"]:
        df[col] = pd.to_datetime(df[col], format="mixed", utc=True)
    return df


df = prepare_dataframe_dates(df)
print("Типи стовпців дати після парсингу:")
print(df[["stations_time", "stations_params_time"]].dtypes)


## 6. 🗄️ Схема бази даних (модель Snowflake)

Очищені дані зберігаються у схемі **сховища даних Snowflake**:

```
DimUnit <-- DimParameter <--+
                             FactMeasurement
               DimStation <--+
```

- **`DimUnit`** -- канонічні одиниці вимірювання
- **`DimParameter`** -- канонічні параметри забруднення / погоди (з підтримкою SCD Type 2)
- **`DimStation`** -- станції моніторингу з GPS-координатами (з підтримкою SCD Type 2)
- **`FactMeasurement`** -- один рядок на сенсорне вимірювання

Стовпці SCD Type 2 (`valid_from`, `valid_to`, `is_current`) дозволяють відстежувати 历史ю перейменувань станцій або переконфігурацій сенсорів без втрати попередніх вимірювань.


In [ ]:
Base = declarative_base()


class DimUnit(Base):
    """Таблиця вимірів: одиниці вимірювання (µg/m³, ppm, °C, …)."""
    __tablename__ = "dim_units"

    unit_key       = Column(Integer, primary_key=True, autoincrement=True)
    unit_name      = Column(String(64), nullable=False)
    unit_symbol    = Column(String(8))
    local_unit_name = Column(String(64))

    # Зворотне посилання: одна одиниця -- багато параметрів
    parameters = relationship("DimParameter", back_populates="unit")


class DimParameter(Base):
    """Таблиця вимірів: виміряні параметри (PM2.5, NO2, Температура, …)."""
    __tablename__ = "dim_parameters"

    parameter_key  = Column(Integer, primary_key=True, autoincrement=True)
    parameter_code = Column(String(64), nullable=False)
    parameter_name = Column(String(64))
    local_name     = Column(String(64))
    unit_key       = Column(Integer, ForeignKey("dim_units.unit_key"))

    # SCD Type 2 -- відстеження змін визначення параметрів з часом
    valid_from = Column(DateTime, nullable=False)
    valid_to   = Column(DateTime)
    is_current = Column(Boolean, default=True)

    unit         = relationship("DimUnit", back_populates="parameters")
    measurements = relationship("FactMeasurement", back_populates="parameter")


class DimStation(Base):
    """Таблиця вимірів: станції моніторингу з GPS-координатами."""
    __tablename__ = "dim_stations"

    station_key     = Column(Integer, primary_key=True, autoincrement=True)
    station_id      = Column(Integer)
    station_name    = Column(String(128))
    latitude        = Column(Float)
    longitude       = Column(Float)
    timezone_offset = Column(Integer)

    # SCD Type 2 -- відстеження перейменувань / переміщень станцій
    valid_from = Column(DateTime, nullable=False)
    valid_to   = Column(DateTime)
    is_current = Column(Boolean, default=True)

    measurements = relationship("FactMeasurement", back_populates="station")


class FactMeasurement(Base):
    """
    Таблиця фактів: один рядок на сенсорне вимірювання.

    Зовнішні ключі посилаються на таблиці вимірів станції, параметра та одиниці.
    Часовий вимір вбудований безпосередньо (без окремої таблиці DimTime).
    """
    __tablename__ = "fact_measurements"

    measurement_id = Column(BigInteger, primary_key=True, autoincrement=True)

    # ── Foreign keys ──────────────────────────────────────────────────────────
    station_key   = Column(Integer, ForeignKey("dim_stations.station_key"),   nullable=False)
    parameter_key = Column(Integer, ForeignKey("dim_parameters.parameter_key"), nullable=False)

    # ── Measured values ───────────────────────────────────────────────────────
    value           = Column(Float)
    quality_ratio   = Column(Float, nullable=True)   # calibration confidence
    pollution_level = Column(Integer)                # optional AQI band

    # ── Time dimension ────────────────────────────────────────────────────────
    measurement_timestamp = Column(DateTime, nullable=False, index=True)
    offset_minutes        = Column(Integer)           # UTC offset of station

    station   = relationship("DimStation",   back_populates="measurements")
    parameter = relationship("DimParameter", back_populates="measurements")


### 6.1 Створення фізичної бази даних

Відкоригуйте значення `host` у `CONFIG` відповідно до середовища розгортання:

| Сценарій | Значення `db_host` |
|---|---|
| Локальний MySQL | `127.0.0.1` |
| Docker Compose | назва сервісу (наприклад, `db`) |
| Віддалений сервер | ім'я хоста або IP |

Облікові дані зчитуються зі змінних середовища -- ніколи не жорстко кодуються.


In [ ]:
def create_db_engine(config: Dict[str, Any]):
    """
    Створює рушій SQLAlchemy з центрального словника CONFIG.

    Parameters
    ----------
    config : Dict[str, Any] -- повинен містити ключі: db_driver, db_host, db_port.
                              Облікові дані: MYSQL_USER, MYSQL_PASSWORD, MYSQL_DATABASE.

    Returns
    -------
    sqlalchemy.engine.Engine
    """
    url = URL.create(
        drivername=config["db_driver"],
        username=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        host=config["db_host"],
        port=config["db_port"],
        database=os.getenv("MYSQL_DATABASE"),
    )
    return create_engine(url)


engine = create_db_engine(CONFIG)
Base.metadata.create_all(engine)   # Створює таблиці, якщо вони ще не існують
print("✅ Схему бази даних створено (або вона вже існує).")


## 7. 🔄 ETL-конвеєр

Кожна функція `transform_and_load_*` дотримується одного шаблону:

1. **Витягти** унікальні записи вимірів з DataFrame
2. **Перевірити** наявність відповідного запису в базі даних (ідемпотентний upsert)
3. **Вставити** нові записи та виконати `flush()` для отримання сурогатних ключів
4. **Повернути** відображення `{бізнес-ключ -> сурогатний_ключ}` для наступного етапу

Всі функції отримують аргумент `session: Session` -- точка впровадження залежностей.


In [ ]:
def transform_and_load_units(df: pd.DataFrame, session: Session) -> Dict[str, int]:
    """
    Виконує upsert унікальних одиниць вимірювання до DimUnit.

    Parameters
    ----------
    df      : pd.DataFrame -- очищений набір даних.
    session : Session      -- активна сесія SQLAlchemy (впроваджена ззовні).

    Returns
    -------
    Dict[str, int] -- відображення {unit_symbol -> unit_key}.
    """
    unique_units = df[["stations_params_unit", "stations_params_localUnit"]].drop_duplicates()
    unit_map: Dict[str, int] = {}

    for _, row in unique_units.iterrows():
        symbol = row["stations_params_unit"]
        unit   = session.query(DimUnit).filter_by(unit_symbol=symbol).first()

        if not unit:
            unit = DimUnit(
                unit_name      = symbol,
                unit_symbol    = symbol,
                local_unit_name= row["stations_params_localUnit"],
            )
            session.add(unit)
            session.flush()   # populate unit.unit_key without committing

        unit_map[unit.unit_symbol] = unit.unit_key

    return unit_map


In [ ]:
def transform_and_load_parameters(
    df      : pd.DataFrame,
    session : Session,
    unit_map: Dict[str, int],
) -> Dict[str, int]:
    """
    Виконує upsert унікальних параметрів вимірювання до DimParameter.

    Parameters
    ----------
    df       : pd.DataFrame   -- очищений набір даних.
    session  : Session        -- активна сесія SQLAlchemy (впроваджена ззовні).
    unit_map : Dict[str, int] -- відображення {unit_symbol -> unit_key}.

    Returns
    -------
    Dict[str, int] -- відображення {parameter_code -> parameter_key}.
    """
    cols = ["stations_params_key", "stations_params_name",
            "stations_params_localName", "stations_params_unit"]
    unique_params = df[cols].drop_duplicates()
    param_map: Dict[str, int] = {}

    for _, row in unique_params.iterrows():
        code_val = row["stations_params_key"]
        param    = session.query(DimParameter).filter_by(parameter_code=code_val).first()

        if not param:
            param = DimParameter(
                parameter_code = code_val,
                parameter_name = row["stations_params_name"],
                local_name     = row["stations_params_localName"],
                unit_key       = unit_map.get(row["stations_params_unit"]),
                valid_from     = datetime.now(),
                is_current     = True,
            )
            session.add(param)
            session.flush()

        param_map[param.parameter_code] = param.parameter_key

    return param_map


In [ ]:
def transform_and_load_stations(df: pd.DataFrame, session: Session) -> Dict[int, int]:
    """
    Виконує upsert унікальних станцій до DimStation.

    Parameters
    ----------
    df      : pd.DataFrame -- очищений набір даних.
    session : Session      -- активна сесія SQLAlchemy (впроваджена ззовні).

    Returns
    -------
    Dict[int, int] -- відображення {бізнес_ід_станції -> сурогатний_ключ_станції}.
    """
    cols = ["stations_id", "stations_name", "Lat", "Long", "stations_offset"]
    unique_stations = df[cols].drop_duplicates()
    station_map: Dict[int, int] = {}

    for _, row in unique_stations.iterrows():
        station = session.query(DimStation).filter_by(
            station_id=row["stations_id"], is_current=True
        ).first()

        if not station:
            station = DimStation(
                station_id     = row["stations_id"],
                station_name   = row["stations_name"],
                latitude       = row["Lat"],
                longitude      = row["Long"],
                timezone_offset= row["stations_offset"],
                valid_from     = datetime.now(),
                is_current     = True,
            )
            session.add(station)
            session.flush()

        station_map[station.station_id] = station.station_key

    return station_map


In [ ]:
def load_fact_measurements(
    df         : pd.DataFrame,
    session    : Session,
    station_map: Dict[int, int],
    param_map  : Dict[str, int],
    batch_size : int = 5000,
) -> None:
    """
    Масово вставляє записи вимірювань до FactMeasurement пакетами.

    Parameters
    ----------
    df          : pd.DataFrame   -- повністю оброблений набір даних.
    session     : Session        -- активна сесія SQLAlchemy (впроваджена ззовні).
    station_map : Dict[int, int] -- {бізнес_ід -> сурогатний_ключ} для станцій.
    param_map   : Dict[str, int] -- {код_параметра -> сурогатний_ключ}.
    batch_size  : int            -- рядків за цикл комміту (за замовчуванням 5 000).
    """
    buffer: List[FactMeasurement] = []
    n_added  = 0
    n_total  = len(df)

    for _, row in df.iterrows():
        ts = row["stations_params_time"]
        if pd.isna(ts):
            continue

        s_key = station_map.get(row["stations_id"])
        p_key = param_map.get(row["stations_params_key"])
        if s_key is None or p_key is None:
            continue   # orphaned record — skip

        buffer.append(FactMeasurement(
            station_key           = s_key,
            parameter_key         = p_key,
            value                 = row["stations_params_value"],
            quality_ratio         = row["stations_params_cr"],
            pollution_level       = row["stations_params_level"],
            measurement_timestamp = ts,
            offset_minutes        = row["stations_params_offset"],
        ))

        if len(buffer) >= batch_size:
            session.bulk_save_objects(buffer)
            session.commit()
            buffer.clear()
            session.expunge_all()   # free ORM identity map memory
            n_added += batch_size
            print(f"\rВставлено: {n_added:,} / {n_total:,}", end="")

    # Flush any remaining records
    if buffer:
        session.bulk_save_objects(buffer)
        session.commit()
        session.expunge_all()
        buffer.clear()

    print(f"\n✅ Завантаження таблиці фактів завершено.")


### 7.1 Оркестратор -- `run_pipeline`

Оркестратор з'єднує чотири функції завантаження у правильному порядку залежностей та обгортає
все в єдину транзакцію: якщо будь-який крок викидає виняток, сесія відкочується.


In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional


# ── Step-function type aliases ────────────────────────────────────────────────
LoadUnitsFn      = Callable[[pd.DataFrame, Session], Dict[str, int]]
LoadParamsFn     = Callable[[pd.DataFrame, Session, Dict[str, int]], Dict[str, int]]
LoadStationsFn   = Callable[[pd.DataFrame, Session], Dict[int, int]]
LoadFactsFn      = Callable[[pd.DataFrame, Session, Dict[int, int], Dict[str, int], int], None]


@dataclass
class ETLPipelineSteps:
    """
    Контейнер для всіх замінюваних ETL-функцій кроків.

    Кожне поле приймає будь-який callable з відповідною сигнатурою, що дозволяє
    легко замінити окремий крок без зміни оркестратора.
    """
    load_units    : LoadUnitsFn    = field(default=transform_and_load_units)
    load_params   : LoadParamsFn   = field(default=transform_and_load_parameters)
    load_stations : LoadStationsFn = field(default=transform_and_load_stations)
    load_facts    : LoadFactsFn    = field(default=load_fact_measurements)


def run_pipeline(
    df         : pd.DataFrame,
    session    : Session,
    batch_size : int              = 5000,
    steps      : Optional[ETLPipelineSteps] = None,
) -> None:
    """
    Оркеструє повне ETL-завантаження у порядку залежностей з повним впровадженням залежностей.

    Порядок завантаження:
      1. Одиниці вимірювання -- без залежностей
      2. Параметри           -- залежать від одиниць
      3. Станції             -- без залежностей
      4. Записи фактів       -- залежать від параметрів і станцій

    Parameters
    ----------
    df         : pd.DataFrame     -- повністю очищений DataFrame.
    session    : Session          -- сесія SQLAlchemy (впроваджена ззовні).
    batch_size : int              -- рядків за цикл комміту.
    steps      : ETLPipelineSteps -- функції кроків (за замовчуванням -- продакшн-реалізації).
    """
    if steps is None:
        steps = ETLPipelineSteps()

    try:
        print("① Завантаження одиниць вимірювання …")
        u_map = steps.load_units(df, session)

        print("② Завантаження параметрів вимірювання …")
        p_map = steps.load_params(df, session, u_map)

        print("③ Завантаження станцій …")
        s_map = steps.load_stations(df, session)

        print("④ Завантаження вимірювань (таблиця фактів) …")
        steps.load_facts(df, session, s_map, p_map, batch_size)

        print("\n🎉 ETL-конвеєр успішно завершено.")

    except Exception as exc:
        session.rollback()
        print(f"\n❌ Конвеєр завершився з помилкою: {exc}")
        raise


# ── Виконання ─────────────────────────────────────────────────────────────────
SessionFactory = sessionmaker(engine)
session = SessionFactory()

# run_pipeline(df, session, batch_size=CONFIG["batch_size"])


## 8. 🔍 Розвідувальний аналіз даних (EDA)

Перед побудовою ML-моделей досліджуємо дані: розподіл, часові паттерни, кореляції та просторовий розподіл.


In [ ]:
# Ensure time column is sorted for all plots
df = df.sort_values("stations_time")

print(df.info())
print("\nВідсутні значення за стовпцями:")
print(df.isnull().sum())


### 8.1 Кількість вимірювань по станціях

Які станції звітують найбільше даних? Станції з дуже малою кількістю записів ненадійні для ML.


In [ ]:
plt.figure(figsize=(10, 8))
df["stations_name"].value_counts().plot(kind="barh")
plt.title("Кількість зафіксованих вимірювань за станцією")
plt.xlabel("Кількість")
plt.tight_layout()
plt.show()


### 8.2 Розподіл значень забруднювачів

Box plot (логарифмічна шкала) виявляє розкид і викиди для кожного забруднювача.


In [ ]:
POLLUTANTS = ["PM2.5", "PM10", "CO2", "Temperature", "Humidity"]

plt.figure(figsize=(15, 8))
sns.boxplot(
    data=df[df["stations_params_key"].isin(POLLUTANTS)],
    x="stations_params_key",
    y="stations_params_value",
)
plt.yscale("log")
plt.title("Розподіл основних забруднювачів (логарифмічна шкала)")
plt.xlabel("Параметр")
plt.ylabel("Значення (лог. шкала)")
plt.tight_layout()
plt.show()


### 8.3 Часовий ряд PM2.5 -- топ-5 станцій

Денні середні роблять графік читабельним, зберігаючи сигнал тренду.


In [ ]:
top_stations = df["stations_name"].value_counts().nlargest(5).index
pm25_data   = df[
    (df["stations_params_key"] == "PM2.5") &
    (df["stations_name"].isin(top_stations))
]

pm25_daily = (
    pm25_data
    .groupby(["stations_name", pd.Grouper(key="stations_time", freq="D")])["stations_params_value"]
    .mean()
    .reset_index()
)

fig = px.line(
    pm25_daily,
    x="stations_time", y="stations_params_value", color="stations_name",
    title="Денні середні значення PM2.5 -- Топ-5 станцій",
    labels={"stations_params_value": "PM2.5 (µg/m³)", "stations_time": "Дата"},
)
fig.show()


### 8.4 Теплова карта кореляцій між забруднювачами

Використовує вибірку 100 000 рядків для управління використанням пам'яті.


In [ ]:
df_sample = df.sample(100_000, random_state=42)

pivot_corr = df_sample.pivot_table(
    index   = ["stations_id", "stations_time"],
    columns = "stations_params_key",
    values  = "stations_params_value",
    aggfunc = "mean",
)

plt.figure(figsize=(12, 10))
sns.heatmap(pivot_corr.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Теплова карта кореляцій -- Забруднювачі та метеофактори")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import plotly.express as px

# 1. Ensure the value column is numeric (convert strings to NaN, then drop or handle them)
df["stations_params_value"] = pd.to_numeric(df["stations_params_value"], errors='coerce')

# 2. Group and aggregate
map_data = (
    df[df["stations_params_key"] == "PM2.5"]
    .groupby("stations_name")
    .agg(
        Lat=("Lat", "first"), 
        Long=("Long", "first"),
        pm25_mean=("stations_params_value", "mean")
    )
    .reset_index()
)

# 3. Double-check: Drop rows where pm25_mean might be NaN to avoid plotting errors
map_data = map_data.dropna(subset=["pm25_mean"])

# 4. Use px.scatter_map (the updated version of scatter_mapbox)
fig = px.scatter_map(
    map_data,
    lat="Lat", 
    lon="Long",
    color="pm25_mean", 
    size="pm25_mean",
    hover_name="stations_name",
    color_continuous_scale=px.colors.sequential.Reds,
    size_max=15, 
    zoom=10,
    map_style="carto-positron", # Note: mapbox_style becomes map_style in the new function
    title="Середня концентрація PM2.5 за розташуванням станції",
)

fig.show()

### 8.7 Добовий паттерн PM2.5 (за годинами доби)


In [ ]:
df["hour"] = df["stations_time"].dt.hour

hourly_pm = (
    df[df["stations_params_key"] == "PM2.5"]
    .groupby("hour")["stations_params_value"]
    .median()
)

plt.figure(figsize=(10, 5))
hourly_pm.plot(kind="line", marker="o", color="teal")
plt.title("Медіанний рівень PM2.5 за годиною доби (добовий паттерн)")
plt.xlabel("Година (UTC)")
plt.ylabel("PM2.5 -- медіана (µg/m³)")
plt.xticks(range(0, 24))
plt.grid(True)
plt.tight_layout()
plt.show()


## 9. 🤖 Конвеєр машинного навчання

### Проектні рішення

| Рішення | Обґрунтування |
|---|---|
| **Ціль:** PM2.5 через 3 години | Достатньо короткий для практичного застосування |
| **Ресемплінг до погодинних даних** | Нерегулярні інтервали порушують часові моделі |
| **Ознаки лагів (1 год -- 72 год)** | Автокореляція -- найсильніший предиктор |
| **Циклічне кодування часу** (sin/cos) | Зберігає кругову природу часу доби |
| **Просторові сусіди** | Сусідні станції -- випереджаючі індикатори |
| **RobustScaler + log1p ціль** | Зменшує вплив екстремальних викидів |
| **TimeSeriesSplit CV** | Запобігає витоку даних з майбутнього |


### 9.1 Ранжування станцій

Відбір станцій з достатньою щільністю даних для надійного навчання.


In [ ]:
def rank_stations(df: pd.DataFrame, min_hours: int = 1000) -> pd.DataFrame:
    """
    Ранжує станції за кількістю валідних погодинних вимірювань PM2.5.

    Parameters
    ----------
    df        : pd.DataFrame -- набір даних у довгому форматі.
    min_hours : int          -- мінімальна кількість валідних вимірювань.

    Returns
    -------
    pd.DataFrame -- ранжована таблиця зі стовпцями [station, count].
    """
    stats = []
    for station in df["stations_name"].unique():
        pm = df[(df["stations_name"] == station) & (df["stations_params_key"] == "PM2.5")]
        if len(pm) >= min_hours:
            stats.append({"station": station, "count": len(pm)})

    return pd.DataFrame(stats).sort_values("count", ascending=False).reset_index(drop=True)


ranking      = rank_stations(df, min_hours=CONFIG["min_station_hours"])
best_station = ranking.iloc[0]["station"]
print(f"Топ станцій:\n{ranking.head(10)}\n")
print(f"-> Обрана станція для аналізу: {best_station}")


### 9.2 Побудова широкої (зведеної) багатостанційної таблиці


In [ ]:
def build_multi_station_table(df: pd.DataFrame, freq: str = "h") -> pd.DataFrame:
    """
    Зводить набір даних у довгому форматі до широкої таблиці та ресемплює до фіксованої частоти.

    Parameters
    ----------
    df   : pd.DataFrame -- очищений набір даних у довгому форматі.
    freq : str          -- частота ресемплінгу (за замовчуванням 'h' = погодинно).

    Returns
    -------
    pd.DataFrame -- широка таблиця з індексом за мітками часу.
    """
    df = df.copy()
    df["stations_params_value"] = pd.to_numeric(df["stations_params_value"], errors="coerce")

    pivot = df.pivot_table(
        index   = "stations_time",
        columns = ["stations_name", "stations_params_key"],
        values  = "stations_params_value",
        aggfunc = "mean",
    )
    return pivot.resample(freq).mean()


pivot = build_multi_station_table(df, freq=CONFIG["resample_freq"])
print("Розміри широкої таблиці:", pivot.shape)


### 9.3 Фільтрація надійних станцій та просторове заповнення пропусків


In [ ]:
def get_pm_matrix(pivot: pd.DataFrame) -> pd.DataFrame:
    """Витягує зріз PM2.5 з мультирівневої зведеної таблиці."""
    return pivot.xs("PM2.5", level=1, axis=1)


def filter_good_stations(
    pm_matrix      : pd.DataFrame,
    min_hours      : int   = 1500,
    min_continuity : float = 0.5,
) -> pd.DataFrame:
    """
    Повертає ранжований DataFrame станцій, що відповідають порогам якості даних.

    Parameters
    ----------
    pm_matrix      : pd.DataFrame -- широка матриця PM2.5.
    min_hours      : int          -- мінімальна кількість не-NaN годин.
    min_continuity : float        -- необхідна частка не-NaN годин (0-1).
    """
    stats = []
    for station in pm_matrix.columns:
        valid      = pm_matrix[station].notna().sum()
        continuity = valid / len(pm_matrix)
        if valid >= min_hours and continuity >= min_continuity:
            stats.append({"station": station, "valid": valid, "continuity": continuity})

    return (
        pd.DataFrame(stats)
        .sort_values(["valid", "continuity"], ascending=False)
        .reset_index(drop=True)
    )


def fill_with_neighbors(pm_matrix: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    """
    Заповнює пропуски в серії PM2.5 зваженим середнім найбільш корельованих сусідів.

    Parameters
    ----------
    pm_matrix : pd.DataFrame -- широка матриця PM2.5.
    top_k     : int          -- кількість найближчих сусідів.

    Returns
    -------
    pd.DataFrame -- копія матриці з заповненими пропусками.
    """
    filled = pm_matrix.copy()
    corr   = pm_matrix.corr()

    for station in pm_matrix.columns:
        neighbors = corr[station].drop(station).sort_values(ascending=False).head(top_k)
        mask      = filled[station].isna()

        for t in pm_matrix.index[mask]:
            vals, weights = [], []
            for nbr, w in neighbors.items():
                val = pm_matrix.loc[t, nbr]
                if not pd.isna(val):
                    vals.append(val)
                    weights.append(max(w, 0.0))   # clip negative correlations

            if vals and sum(weights) > 0:
                filled.loc[t, station] = np.average(vals, weights=weights)

    return filled


pm_matrix = get_pm_matrix(pivot)
stats_df  = filter_good_stations(
    pm_matrix,
    min_hours      = CONFIG["min_station_hours"],
    min_continuity = CONFIG["min_continuity"],
)
pm_filled = fill_with_neighbors(pm_matrix[stats_df["station"]], top_k=CONFIG["top_k_neighbors"])

print("Якісні станції:\n", stats_df.head(10))


### 9.4 Інженерія ознак

Найважливіший крок для продуктивності моделі. Створюються:
- **Щільні ознаки лагів** -- значення PM2.5 за t-1 год ... t-24 год, t-48 год, t-72 год
- **Ковзаючі статистики** -- середнє та стандартне відхилення за 3 год, 6 год, 12 год
- **Циклічні часові ознаки** -- sin/cos кодування години та дня тижня
- **Метеорологічна динаміка** -- дельта та ковзаюче середнє для Температури, Вологості, Тиску
- **Просторовий контекст** -- зсунуті значення PM2.5 з корельованих сусідніх станцій


In [ ]:
def select_stations(
    stats_df   : pd.DataFrame,
    pm_filled  : pd.DataFrame,
    top_k      : int = 3,
) -> Tuple[str, List[str]]:
    """
    Обирає основну цільову станцію та найбільш корельованих сусідів.

    Parameters
    ----------
    stats_df  : pd.DataFrame -- ранжована таблиця якості станцій.
    pm_filled : pd.DataFrame -- матриця PM2.5 із заповненими пропусками.
    top_k     : int          -- кількість сусідів для включення.

    Returns
    -------
    Tuple[str, List[str]] -- (назва_цільової_станції, [назви_сусідів]).
    """
    target   = stats_df.iloc[0]["station"]
    corr     = pm_filled.corr()[target].drop(target)
    related  = corr.sort_values(ascending=False).head(top_k).index.tolist()
    return target, related


def build_features(
    pivot         : pd.DataFrame,
    pm_filled     : pd.DataFrame,
    target        : str,
    related       : List[str],
    forecast_h    : int  = 3,
    include_pm10  : bool = False,
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Будує матрицю ознак X та вектор цілі y для ML-моделей.

    Parameters
    ----------
    pivot        : pd.DataFrame -- повна широка таблиця.
    pm_filled    : pd.DataFrame -- матриця PM2.5 із заповненими пропусками.
    target       : str          -- назва цільової станції.
    related      : List[str]    -- назви сусідніх станцій.
    forecast_h   : int          -- горизонт прогнозу в годинах.
    include_pm10 : bool         -- чи включати PM10 як ознаку.

    Returns
    -------
    (X, y) : Tuple[pd.DataFrame, pd.Series]
    """
    y = pm_filled[target]

    # Start with all meteorological columns for the target station
    X = pivot[target].copy()
    if not include_pm10:
        X = X.drop(columns=["PM10"], errors="ignore")
    X = X.drop(columns=["PM2.5"], errors="ignore")   # prevent target leakage

    # ── Shift target forward in time (predicting the future) ─────────────────
    y_target      = y.shift(-forecast_h)
    y_target.name = "target"

    # ── Lag features ──────────────────────────────────────────────────────────
    for lag in list(range(1, 25)) + [48, 72]:
        X[f"pm_lag_{lag}h"] = y.shift(lag)

    # ── Rolling statistics ────────────────────────────────────────────────────
    X["pm_roll_mean_3h"]  = y.shift(1).rolling(3).mean()
    X["pm_roll_mean_6h"]  = y.shift(1).rolling(6).mean()
    X["pm_roll_mean_12h"] = y.shift(1).rolling(12).mean()
    X["pm_roll_std_6h"]   = y.shift(1).rolling(6).std()

    # ── Cyclical time encoding ────────────────────────────────────────────────
    X["hour_sin"] = np.sin(2 * np.pi * X.index.hour / 24.0)
    X["hour_cos"] = np.cos(2 * np.pi * X.index.hour / 24.0)
    X["dow_sin"]  = np.sin(2 * np.pi * X.index.dayofweek / 7.0)
    X["dow_cos"]  = np.cos(2 * np.pi * X.index.dayofweek / 7.0)

    # ── Meteorological dynamics ───────────────────────────────────────────────
    for met_col in ["Temperature", "Humidity", "Pressure"]:
        if met_col in X.columns:
            X[f"{met_col}_delta_3h"]  = X[met_col].diff(3)
            X[f"{met_col}_delta_6h"]  = X[met_col].diff(6)
            X[f"{met_col}_roll_12h"]  = X[met_col].rolling(12).mean()

    if "Temperature" in X.columns and "Humidity" in X.columns:
        X["temp_x_humidity"] = X["Temperature"] * X["Humidity"]

    # ── Spatial context (neighbouring stations) ───────────────────────────────
    for nbr in related:
        X[f"{nbr}_pm_lag_1h"] = pm_filled[nbr].shift(1)

    # ── Final clean-up ────────────────────────────────────────────────────────
    combined = pd.concat([X, y_target], axis=1)
    combined = combined[combined["target"].notna()]
    combined = combined.ffill().bfill()

    return combined.drop(columns=["target"]), combined["target"]


target_station, related_stations = select_stations(
    stats_df, pm_filled, top_k=CONFIG["top_k_neighbors"]
)

X, y = build_features(
    pivot, pm_filled, target_station, related_stations,
    forecast_h   = CONFIG["forecast_horizon_h"],
    include_pm10 = CONFIG["use_pm10_feature"],
)

print(f"Цільова станція  : {target_station}")
print(f"Сусідні станції  : {related_stations}")
print(f"Матриця ознак    : {X.shape}")


### 9.5 Розподіл на навчальну та тестову вибірки (хронологічний)


In [ ]:
def time_split(
    X         : pd.DataFrame,
    y         : pd.Series,
    test_size : float = 0.2,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Розподіляє дані хронологічно для запобігання витоку майбутніх даних.

    Parameters
    ----------
    X, y      : матриця ознак і вектор цілі (з індексом часу).
    test_size : float -- частка даних для тестування.

    Returns
    -------
    X_train, X_test, y_train, y_test
    """
    split = int(len(X) * (1 - test_size))
    return X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]


X_train, X_test, y_train, y_test = time_split(X, y, test_size=CONFIG["test_size"])
tscv = TimeSeriesSplit(n_splits=CONFIG["cv_splits"])

print(f"Навчальна вибірка: {len(X_train):,} рядків  |  Тестова: {len(X_test):,} рядків")


## 10. 🏋️ Навчання моделей

Кожна модель обгортається в конвеєр `TransformedTargetRegressor`:

```
RobustScaler -> [Модель] -> log1p(y) / expm1(y_hat)
```

- **RobustScaler** -- використовує медіану/IQR, тому викиди не спотворюють масштабування
- **Трансформація log1p** -- стискає правий хвіст розподілу PM2.5
- **GridSearchCV з TimeSeriesSplit** -- налаштовує гіперпараметри без витоку з майбутнього


In [ ]:
def train_tuned_models(
    X_train : pd.DataFrame,
    y_train : pd.Series,
    tscv    : TimeSeriesSplit,
) -> Dict[str, Any]:
    """
    Навчає та налаштовує Ridge, XGBoost і RandomForest через GridSearchCV,
    потім збирає ансамбль VotingRegressor з найкращих версій.

    Parameters
    ----------
    X_train : pd.DataFrame    -- навчальні ознаки.
    y_train : pd.Series       -- вектор цілі.
    tscv    : TimeSeriesSplit -- розбивач крос-валідації (впроваджений ззовні).

    Returns
    -------
    Dict[str, Any] -- {назва_моделі: навчений estimator}.
    """
    def _wrap(model_obj):
        """Обгортає модель у стандартний конвеєр масштабування + лог-трансформації."""
        return TransformedTargetRegressor(
            regressor     = Pipeline([("scaler", RobustScaler()), ("model", model_obj)]),
            func          = np.log1p,
            inverse_func  = np.expm1,
        )

    # ── 1. Ridge Regression ───────────────────────────────────────────────────
    grid_ridge = GridSearchCV(
        _wrap(Ridge()),
        {"regressor__model__alpha": [0.1, 1.0, 10.0, 100.0]},
        cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1,
    )

    # ── 2. XGBoost ────────────────────────────────────────────────────────────
    grid_xgb = GridSearchCV(
        _wrap(xgb.XGBRegressor(objective="reg:squarederror", random_state=42, verbosity=0)),
        {
            "regressor__model__n_estimators": [100, 300],
            "regressor__model__max_depth"   : [3, 5],
            "regressor__model__learning_rate": [0.01, 0.1],
        },
        cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1,
    )

    # ── 3. Random Forest ──────────────────────────────────────────────────────
    grid_rf = GridSearchCV(
        _wrap(RandomForestRegressor(random_state=42)),
        {
            "regressor__model__n_estimators"  : [100, 200],
            "regressor__model__max_depth"     : [10, 20, None],
            "regressor__model__min_samples_leaf": [1, 4],
        },
        cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1,
    )

    print("Налаштування гребеневої регресії …")
    grid_ridge.fit(X_train, y_train)

    print("Налаштування XGBoost …")
    grid_xgb.fit(X_train, y_train)

    print("Налаштування випадкового лісу …")
    grid_rf.fit(X_train, y_train)

    # ── Ensemble from best individual models ──────────────────────────────────
    best_ridge = grid_ridge.best_estimator_.regressor_.named_steps["model"]
    best_xgb   = grid_xgb.best_estimator_.regressor_.named_steps["model"]
    best_rf    = grid_rf.best_estimator_.regressor_.named_steps["model"]

    ensemble_pipe = _wrap(VotingRegressor([
        ("ridge", best_ridge),
        ("xgb",   best_xgb),
        ("rf",    best_rf),
    ]))
    print("Навчання ансамблю …")
    ensemble_pipe.fit(X_train, y_train)

    print("\n✅ Всі моделі навчено.")
    return {
        "Ridge_Tuned"     : grid_ridge.best_estimator_,
        "XGBoost_Tuned"   : grid_xgb.best_estimator_,
        "RandomForest_Tuned": grid_rf.best_estimator_,
        "Ensemble"        : ensemble_pipe,
    }


models = train_tuned_models(X_train, y_train, tscv)


### 9.6 Оркестратор ML-конвеєра -- `run_ml_pipeline`

Оркестратор з впровадженням залежностей для повного ML-конвеєра.

> **Виправлення:** поле `evaluate` у `MLPipelineSteps` тепер має значення `None` за замовчуванням.
> Функція `evaluate` визначена нижче (розділ 11); `field(default=evaluate)` спричиняло `NameError`
> під час завантаження цієї клітини, що призводило до зависання виконання. Тепер оркестратор
> підставляє `evaluate` динамічно під час виклику.


In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple


# ── Аліаси типів для функцій-кроків ──────────────────────────────────────────
RankStationsFn    = Callable[[pd.DataFrame, int], pd.DataFrame]
BuildTableFn      = Callable[[pd.DataFrame, str], pd.DataFrame]
FilterStationsFn  = Callable[[pd.DataFrame, int, float], pd.DataFrame]
FillGapsFn        = Callable[[pd.DataFrame, int], pd.DataFrame]
SelectStationsFn  = Callable[[pd.DataFrame, pd.DataFrame, int], Tuple[str, list]]
BuildFeaturesFn   = Callable[..., Tuple[pd.DataFrame, pd.Series]]
SplitFn           = Callable[[pd.DataFrame, pd.Series, float], Tuple]
TrainFn           = Callable[[pd.DataFrame, pd.Series, "TimeSeriesSplit"], Dict[str, Any]]
EvaluateFn        = Callable[[Dict[str, Any], pd.DataFrame, pd.Series], pd.DataFrame]


@dataclass
class MLPipelineSteps:
    """
    Контейнер для всіх замінюваних функцій-кроків ML-конвеєра.

    Поле ``evaluate`` має значення None за замовчуванням, оскільки функція
    ``evaluate`` визначена у розділі 11 (нижче в ноутбуку). Використання
    ``field(default=evaluate)`` спричинило б NameError при виконанні цієї
    клітини, що призводило до її зависання без повернення результату.
    Оркестратор ``run_ml_pipeline`` підставляє правильне значення динамічно.
    """
    rank_stations    : RankStationsFn    = field(default=rank_stations)
    build_table      : BuildTableFn      = field(default=build_multi_station_table)
    filter_stations  : FilterStationsFn  = field(default=filter_good_stations)
    fill_gaps        : FillGapsFn        = field(default=fill_with_neighbors)
    select_stations  : SelectStationsFn  = field(default=select_stations)
    build_features   : BuildFeaturesFn   = field(default=build_features)
    split            : SplitFn           = field(default=time_split)
    train            : TrainFn           = field(default=train_tuned_models)
    # evaluate = None за замовчуванням: функція визначена нижче в ноутбуку
    evaluate         : Optional[EvaluateFn] = field(default=None)


@dataclass
class MLPipelineResult:
    """Типізований контейнер, що повертається ``run_ml_pipeline``."""
    models          : Dict[str, Any]
    metrics         : pd.DataFrame
    X_train         : pd.DataFrame
    X_test          : pd.DataFrame
    y_train         : pd.Series
    y_test          : pd.Series
    target_station  : str
    related_stations: list


def run_ml_pipeline(
    df     : pd.DataFrame,
    config : Dict[str, Any],
    steps  : Optional[MLPipelineSteps] = None,
) -> MLPipelineResult:
    """
    Оркеструє повний ML-конвеєр з повним впровадженням залежностей.

    Порядок етапів
    --------------
    1. rank_stations   -- оцінює станції за щільністю даних PM2.5
    2. build_table     -- зводить дані до широкої погодинної таблиці
    3. filter_stations -- відкидає станції нижче порогів якості
    4. fill_gaps       -- просторово заповнює пропуски через сусідів
    5. select_stations -- обирає цільову станцію та сусідів
    6. build_features  -- будує матрицю ознак X та вектор цілі y
    7. split           -- хронологічний розподіл train/test
    8. train           -- навчає та налаштовує всі моделі
    9. evaluate        -- обчислює R2, MAE, RMSE на тестовій вибірці

    Parameters
    ----------
    df     : pd.DataFrame      -- очищений набір даних у довгому форматі.
    config : Dict[str, Any]    -- центральний словник CONFIG.
    steps  : MLPipelineSteps   -- функції кроків (за замовчуванням -- продакшн).

    Returns
    -------
    MLPipelineResult -- контейнер з моделями, метриками та даними.
    """
    if steps is None:
        steps = MLPipelineSteps()

    # Підставляємо evaluate динамічно (вирішення проблеми порядку визначення)
    eval_fn = steps.evaluate if steps.evaluate is not None else evaluate

    tscv = TimeSeriesSplit(n_splits=config["cv_splits"])

    print("① Ранжування станцій ...")
    ranking = steps.rank_stations(df, min_hours=config["min_station_hours"])
    print(f"   -> {len(ranking)} станцій відповідають критеріям")

    print("② Побудова широкої погодинної таблиці ...")
    pivot = steps.build_table(df, freq=config["resample_freq"])
    print(f"   -> розміри: {pivot.shape}")

    print("③ Фільтрація надійних станцій ...")
    pm_matrix = get_pm_matrix(pivot)
    stats_df  = steps.filter_stations(
        pm_matrix,
        min_hours      = config["min_station_hours"],
        min_continuity = config["min_continuity"],
    )
    print(f"   -> {len(stats_df)} станцій пройшли фільтр")

    print("④ Заповнення пропусків через просторових сусідів ...")
    pm_filled = steps.fill_gaps(pm_matrix[stats_df["station"]], top_k=config["top_k_neighbors"])

    print("⑤ Вибір цільової та сусідніх станцій ...")
    target, related = steps.select_stations(stats_df, pm_filled, top_k=config["top_k_neighbors"])
    print(f"   -> ціль: {target}")
    print(f"   -> сусіди: {related}")

    print("⑥ Інженерія ознак ...")
    X, y = steps.build_features(
        pivot, pm_filled, target, related,
        forecast_h   = config["forecast_horizon_h"],
        include_pm10 = config["use_pm10_feature"],
    )
    print(f"   -> матриця ознак: {X.shape}")

    print("⑦ Розподіл на навчальну / тестову вибірки ...")
    X_train, X_test, y_train, y_test = steps.split(X, y, test_size=config["test_size"])
    print(f"   -> навчальна: {len(X_train):,}  |  тестова: {len(X_test):,}")

    print("⑧ Навчання моделей ...")
    models = steps.train(X_train, y_train, tscv)

    print("⑨ Оцінювання ...")
    metrics = eval_fn(models, X_test, y_test)
    print(metrics)

    print("\n🎉 ML-конвеєр успішно завершено.")
    return MLPipelineResult(
        models           = models,
        metrics          = metrics,
        X_train          = X_train,
        X_test           = X_test,
        y_train          = y_train,
        y_test           = y_test,
        target_station   = target,
        related_stations = related,
    )


# ── Виконання ─────────────────────────────────────────────────────────────────
# result = run_ml_pipeline(df, CONFIG)
# plot_error_analysis(result.models, result.X_test, result.y_test)
# plot_feature_importance(result.models, result.X_train)
# plot_time_series_comparison(result.models, result.X_test, result.y_test)


## 11. 📊 Оцінювання та візуалізація


In [ ]:
def evaluate(models: Dict[str, Any], X_test: pd.DataFrame, y_test: pd.Series) -> pd.DataFrame:
    """
    Обчислює R2, MAE та RMSE для кожної моделі на тестовій вибірці.

    Parameters
    ----------
    models : Dict[str, Any] -- {назва: навчений estimator}.
    X_test : pd.DataFrame   -- тестові ознаки.
    y_test : pd.Series      -- справжні цільові значення.

    Returns
    -------
    pd.DataFrame -- таблиця метрик, відсортована за спаданням R2.
    """
    rows = []
    for name, model in models.items():
        preds = model.predict(X_test)
        rows.append({
            "Model": name,
            "R²"  : round(r2_score(y_test, preds), 4),
            "MAE" : round(mean_absolute_error(y_test, preds), 3),
            "RMSE": round(np.sqrt(mean_squared_error(y_test, preds)), 3),
        })
    return pd.DataFrame(rows).set_index("Model").sort_values("R²", ascending=False)


results = evaluate(models, X_test, y_test)
print("\n── Продуктивність моделей ─────────────────────────")
print(results)


### 11.1 Реальне vs прогнозоване та розподіл залишків


In [ ]:
def plot_error_analysis(
    models : Dict[str, Any],
    X_test : pd.DataFrame,
    y_test : pd.Series,
) -> None:
    """
    Парний графік: діаграма розсіювання реальне vs прогнозоване та гістограма залишків.

    Parameters
    ----------
    models : Dict[str, Any] -- повинен містити ключ 'Ensemble'.
    X_test : pd.DataFrame
    y_test : pd.Series
    """
    model = models["Ensemble"]
    preds = model.predict(X_test)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # ── Scatter: actual vs predicted ─────────────────────────────────────────
    sns.scatterplot(x=y_test, y=preds, alpha=0.4, ax=ax1)
    lims = [y_test.min(), y_test.max()]
    ax1.plot(lims, lims, "r--", label="Ідеальний прогноз")
    ax1.set_title(
        f"Реальне vs Прогнозоване PM2.5 (Ансамбль)\n"
        f"MAE = {mean_absolute_error(y_test, preds):.2f} µg/m³"
    )
    ax1.set_xlabel("Реальне PM2.5")
    ax1.set_ylabel("Прогнозоване PM2.5")
    ax1.legend()
    ax1.grid(alpha=0.3)

    # ── Histogram: residuals ──────────────────────────────────────────────────
    residuals = y_test - preds
    sns.histplot(residuals, kde=True, color="purple", ax=ax2)
    ax2.axvline(0, color="black", linestyle="--")
    ax2.set_title("Розподіл залишків")
    ax2.set_xlabel("Похибка (Реальне - Прогнозоване)")
    ax2.set_ylabel("Частота")

    plt.tight_layout()
    plt.savefig("error_analysis.png", dpi=150)
    plt.show()


plot_error_analysis(models, X_test, y_test)


### 11.2 Важливість ознак (XGBoost)


In [ ]:
def plot_feature_importance(
    models  : Dict[str, Any],
    X_train : pd.DataFrame,
    top_n   : int = 15,
) -> None:
    """
    Стовпчаста діаграма топ-N найважливіших ознак налаштованої моделі XGBoost.

    Parameters
    ----------
    models  : Dict[str, Any] -- повинен містити ключ 'XGBoost_Tuned'.
    X_train : pd.DataFrame   -- для отримання назв стовпців.
    top_n   : int            -- кількість ознак для відображення.
    """
    xgb_model   = models["XGBoost_Tuned"].regressor_.named_steps["model"]
    importances = xgb_model.feature_importances_

    fi_df = (
        pd.DataFrame({"Feature": X_train.columns, "Importance": importances})
        .sort_values("Importance", ascending=False)
        .head(top_n)
    )

    plt.figure(figsize=(10, 8))
    sns.barplot(data=fi_df, x="Importance", y="Feature", palette="viridis")
    plt.title(f"Топ-{top_n} найважливіших ознак (XGBoost)")
    plt.xlabel("Відносна оцінка важливості")
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150)
    plt.show()


plot_feature_importance(models, X_train)


### 11.3 Порівняння прогнозів часового ряду -- фінальний тиждень


In [ ]:
def plot_time_series_comparison(
    models     : Dict[str, Any],
    X_test     : pd.DataFrame,
    y_test     : pd.Series,
    last_n_h   : int = 168,
) -> None:
    """
    Лінійний графік реальних значень vs прогнозів ансамблю за фінальні N годин тестової вибірки.

    Parameters
    ----------
    models   : Dict[str, Any] -- повинен містити ключ 'Ensemble'.
    X_test   : pd.DataFrame
    y_test   : pd.Series
    last_n_h : int            -- кількість годин (за замовчуванням 168 = 1 тиждень).
    """
    preds = models["Ensemble"].predict(X_test)

    plt.figure(figsize=(15, 5))
    plt.plot(y_test.index[-last_n_h:], y_test.values[-last_n_h:],
             label="Реальне", alpha=0.8)
    plt.plot(y_test.index[-last_n_h:], preds[-last_n_h:],
             label="Ансамбль (прогноз)", linestyle="--"
    plt.title(f"Фінальний тиждень -- Реальне vs Прогнозоване PM2.5 ({CONFIG['forecast_horizon_h']} год вперед)")
    plt.xlabel("Час")
    plt.ylabel("PM2.5 (µg/m³)")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig("timeseries_performance.png", dpi=150)
    plt.show()


plot_time_series_comparison(models, X_test, y_test, last_n_h=168)
